# 📊 Base vs Fine-tuned MobileVLM 비교 평가

## 평가 전략
단순 ROUGE/BLEU 대신, **실제 사용 목적(1문장 환자 상태 보고서)** 에 맞는 3가지 지표로 비교:

| 지표 | 설명 |
|---|---|
| **포맷 준수율** | 4-class 중 하나를 포함한 1문장 형식 지키는 비율 |
| **프롬프트 의존도** | 짧은 프롬프트에서 형식 유지 가능 여부 |
| **응답 일관성** | 동일 이미지 5회 반복 시 출력 안정성 |

## 🔧 Cell 1: 환경 설치 및 설정

In [ ]:
# ── 필수 패키지 설치 ──────────────────────────────────────────
!pip install -q transformers==4.40.0 peft==0.9.0 bitsandbytes>=0.44.0 accelerate
!pip install -q rouge-score nltk matplotlib pillow pandas

# ── MobileVLM 커스텀 모델 코드 클론 ────────────────────────────
import os
if not os.path.exists('/content/MobileVLM'):
    !git clone -q https://github.com/Meituan-AutoML/MobileVLM /content/MobileVLM

import sys
sys.path.insert(0, '/content/MobileVLM')
print('✅ 환경 설치 완료')

## 📁 Cell 2: Google Drive 마운트 & 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── 경로 설정 (필요시 수정) ────────────────────────────────────
DRIVE_BASE   = '/content/drive/MyDrive/fall_dataset'
IMAGE_ZIP    = f'{DRIVE_BASE}/images.zip'
LABEL_DIR    = f'{DRIVE_BASE}/labels_vlm'
ADAPTER_DIR  = f'{DRIVE_BASE}/mobilevlm_qlora_adapter'
RESULT_DIR   = f'{DRIVE_BASE}/evaluation_results'
IMAGE_DIR    = '/content/images'

os.makedirs(RESULT_DIR, exist_ok=True)

# ── 이미지 압축 해제 ──────────────────────────────────────────
if not os.path.exists(IMAGE_DIR):
    import zipfile
    print('이미지 압축 해제 중...')
    with zipfile.ZipFile(IMAGE_ZIP, 'r') as z:
        z.extractall('/content')

print(f'✅ 이미지 폴더: {len(os.listdir(IMAGE_DIR))}장')
print(f'✅ 라벨 폴더: {len(os.listdir(LABEL_DIR))}개')

## 🖼️ Cell 3: 샘플 이미지 선택 (카테고리별 균형)

In [ ]:
import json
from pathlib import Path

# I001~I003: bed_exit, I004~I007: moving, I008~I010: fall
CATEGORY_MAP = {
    'bed_exit': ['I001', 'I002', 'I003'],
    'moving':   ['I004', 'I005', 'I006', 'I007'],
    'fall':     ['I008', 'I009', 'I010'],
}
SAMPLES_PER_CAT = {'bed_exit': 3, 'moving': 3, 'fall': 4}  # total 10

# 4-class 키워드 (정답 판별용)
VALID_KEYWORDS = ['낙상', '침대이탈', '이탈', '이동', '휴식']

def get_category(filename):
    """파일명에서 I-index를 보고 카테고리 반환"""
    icode = filename.split('_')[-1].replace('.JPG','').replace('.json','')
    for cat, codes in CATEGORY_MAP.items():
        if icode in codes:
            return cat
    return 'moving'  # fallback

def load_gt_caption(label_path):
    """GT 첫 번째 캡션 반환"""
    with open(label_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data['vlm_labels']['captions'][0]

# 샘플 선택
label_files = sorted(Path(LABEL_DIR).glob('*.json'))
sample_pool = {'bed_exit': [], 'moving': [], 'fall': []}

for lf in label_files:
    cat = get_category(lf.name)
    img_name = lf.stem + '.JPG'
    img_path = os.path.join(IMAGE_DIR, img_name)
    if os.path.exists(img_path) and len(sample_pool[cat]) < SAMPLES_PER_CAT[cat]:
        sample_pool[cat].append({
            'image': img_path,
            'label': str(lf),
            'category': cat,
            'gt_caption': load_gt_caption(str(lf))
        })

SAMPLES = sample_pool['bed_exit'] + sample_pool['moving'] + sample_pool['fall']
print(f'✅ 샘플 선택: {len(SAMPLES)}장')
for s in SAMPLES:
    print(f"  [{s['category']:10s}] {os.path.basename(s['image'])}")

## 🤖 Cell 4: 모델 로드 헬퍼 함수

In [ ]:
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig
from PIL import Image

# MobileVLM 관련 임포트
from mobilevlm.model.mobilevlm import load_pretrained_model
from mobilevlm.conversation import conv_templates
from mobilevlm.utils import disable_torch_init

BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

MODEL_NAME = 'mtgv/MobileVLM_V2-1.7B'

# ── 프롬프트 정의 ──────────────────────────────────────────────
SHORT_PROMPT = "현재 환자의 상태를 보고하세요."

LONG_PROMPT = (
    "병원 병동 CCTV 영상에서 환자의 상태를 관찰하고 있습니다. "
    "아래 4가지 상태 중 하나를 판단하여 한국어로 1문장으로 보고하세요:\n"
    "- 낙상 발생 (환자가 바닥에 쓰러진 경우)\n"
    "- 침대 이탈 (침대에서 내려오는 경우)\n"
    "- 이동 중 (걷거나 이동하는 경우)\n"
    "- 침대에서 휴식 중 (누워있거나 앉아있는 경우)\n"
    "현재 환자의 상태:"
)

def run_mobilevlm_inference(model, tokenizer, image_processor, image_path, prompt, num_runs=1):
    """MobileVLM 추론 실행, num_runs번 반복하여 출력 리스트 반환"""
    from mobilevlm.utils import process_images, tokenizer_image_token
    from mobilevlm.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN

    image = Image.open(image_path).convert('RGB')
    image_tensor = process_images([image], image_processor, model.config)[0]
    image_tensor = image_tensor.to(model.device, dtype=torch.float16)

    full_prompt = DEFAULT_IMAGE_TOKEN + '\n' + prompt
    conv = conv_templates['v1'].copy()
    conv.append_message(conv.roles[0], full_prompt)
    conv.append_message(conv.roles[1], None)
    prompt_str = conv.get_prompt()

    input_ids = tokenizer_image_token(
        prompt_str, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt'
    ).unsqueeze(0).to(model.device)

    outputs = []
    with torch.inference_mode():
        for _ in range(num_runs):
            output_ids = model.generate(
                input_ids,
                images=image_tensor.unsqueeze(0),
                do_sample=True if num_runs > 1 else False,
                temperature=0.7 if num_runs > 1 else 1.0,
                max_new_tokens=64,
                use_cache=True,
            )
            text = tokenizer.decode(output_ids[0, input_ids.shape[1]:], skip_special_tokens=True).strip()
            outputs.append(text)
    return outputs

def check_format_compliance(text):
    """4-class 키워드 포함 여부 및 1문장인지 확인"""
    has_keyword = any(kw in text for kw in VALID_KEYWORDS)
    is_short = len(text) < 60 and text.count('.') <= 1
    return has_keyword and is_short

print('✅ 헬퍼 함수 정의 완료')
print(f'SHORT_PROMPT ({len(SHORT_PROMPT)}자): {SHORT_PROMPT}')
print(f'LONG_PROMPT  ({len(LONG_PROMPT)}자): {LONG_PROMPT[:60]}...')

## 📋 Cell 5: Base 모델 로드

In [ ]:
disable_torch_init()
print('Base 모델 로드 중... (최초 실행 시 HuggingFace 다운로드 ~3GB)')

tokenizer_base, model_base, image_processor_base, _ = load_pretrained_model(
    model_path=MODEL_NAME,
    model_base=None,
    model_name='MobileVLM_V2-1.7B',
    load_8bit=False,
    load_4bit=True,
)
model_base.eval()
print('✅ Base 모델 로드 완료')

## 🧪 Cell 6: [실험 1] 포맷 준수율 — Base 모델 (짧은 프롬프트 vs 긴 프롬프트)

In [ ]:
print('=' * 60)
print('실험 1: Base 모델 포맷 준수율 측정')
print('=' * 60)

results_base = []

for i, sample in enumerate(SAMPLES):
    img_path = sample['image']
    cat = sample['category']
    fname = os.path.basename(img_path)

    # 짧은 프롬프트
    short_out = run_mobilevlm_inference(
        model_base, tokenizer_base, image_processor_base, img_path, SHORT_PROMPT
    )[0]

    # 긴 프롬프트
    long_out = run_mobilevlm_inference(
        model_base, tokenizer_base, image_processor_base, img_path, LONG_PROMPT
    )[0]

    results_base.append({
        'image': fname,
        'category': cat,
        'short_output': short_out,
        'short_comply': check_format_compliance(short_out),
        'long_output': long_out,
        'long_comply': check_format_compliance(long_out),
    })

    print(f'[{i+1:02d}/{len(SAMPLES)}] {fname} ({cat})')
    print(f'  SHORT: {short_out} → {"✅" if results_base[-1]["short_comply"] else "❌"}')
    print(f'  LONG:  {long_out} → {"✅" if results_base[-1]["long_comply"] else "❌"}')
    print()

base_short_rate = sum(r['short_comply'] for r in results_base) / len(results_base) * 100
base_long_rate  = sum(r['long_comply']  for r in results_base) / len(results_base) * 100
print(f'Base 모델 포맷 준수율 — 짧은 프롬프트: {base_short_rate:.0f}%')
print(f'Base 모델 포맷 준수율 — 긴 프롬프트:  {base_long_rate:.0f}%')

## 🔬 Cell 7: Fine-tuned 모델 로드 (Base + LoRA Adapter)

In [ ]:
from peft import PeftModel

# Base를 재활용하거나, 필요시 새로 로드
# LoRA adapter 적용
print('Fine-tuned 모델 로드 중 (Base + LoRA adapter)...')
model_ft = PeftModel.from_pretrained(model_base, ADAPTER_DIR)
model_ft.eval()
print('✅ Fine-tuned 모델 로드 완료')

# tokenizer, image_processor는 base와 동일 사용
tokenizer_ft = tokenizer_base
image_processor_ft = image_processor_base

## 🧪 Cell 8: [실험 1] 포맷 준수율 — Fine-tuned 모델 (짧은 프롬프트)

In [ ]:
print('=' * 60)
print('실험 1: Fine-tuned 모델 포맷 준수율 측정')
print('=' * 60)

results_ft = []

for i, sample in enumerate(SAMPLES):
    img_path = sample['image']
    cat = sample['category']
    fname = os.path.basename(img_path)

    # Fine-tuned는 짧은 프롬프트만 테스트 (핵심 어필)
    short_out = run_mobilevlm_inference(
        model_ft, tokenizer_ft, image_processor_ft, img_path, SHORT_PROMPT
    )[0]

    results_ft.append({
        'image': fname,
        'category': cat,
        'short_output': short_out,
        'short_comply': check_format_compliance(short_out),
    })

    print(f'[{i+1:02d}/{len(SAMPLES)}] {fname} ({cat})')
    print(f'  SHORT: {short_out} → {"✅" if results_ft[-1]["short_comply"] else "❌"}')
    print()

ft_short_rate = sum(r['short_comply'] for r in results_ft) / len(results_ft) * 100
print(f'Fine-tuned 모델 포맷 준수율 — 짧은 프롬프트: {ft_short_rate:.0f}%')
print(f'\n비교 요약:')
print(f'  Base (짧은 프롬프트): {base_short_rate:.0f}%')
print(f'  Base (긴 프롬프트):   {base_long_rate:.0f}%')
print(f'  Fine-tuned (짧은):    {ft_short_rate:.0f}%  ← 프롬프트 없이도 준수!')

## 🧪 Cell 9: [실험 2] 응답 일관성 — 동일 이미지 5회 반복

In [ ]:
print('=' * 60)
print('실험 2: 응답 일관성 (동일 이미지 5회 반복 추론)')
print('=' * 60)

# 카테고리별 대표 이미지 1장씩 (총 3장) 선택
CONSISTENCY_SAMPLES = [
    SAMPLES[0],   # bed_exit 대표
    SAMPLES[3],   # moving 대표
    SAMPLES[6],   # fall 대표
]
N_RUNS = 5

consistency_results = []

for sample in CONSISTENCY_SAMPLES:
    img_path = sample['image']
    cat = sample['category']
    fname = os.path.basename(img_path)

    # Base 모델 5회 반복 (짧은 프롬프트)
    base_runs = run_mobilevlm_inference(
        model_base, tokenizer_base, image_processor_base,
        img_path, SHORT_PROMPT, num_runs=N_RUNS
    )

    # Fine-tuned 모델 5회 반복 (짧은 프롬프트)
    ft_runs = run_mobilevlm_inference(
        model_ft, tokenizer_ft, image_processor_ft,
        img_path, SHORT_PROMPT, num_runs=N_RUNS
    )

    base_unique = len(set(base_runs))
    ft_unique   = len(set(ft_runs))

    consistency_results.append({
        'image': fname,
        'category': cat,
        'base_runs': base_runs,
        'ft_runs': ft_runs,
        'base_unique': base_unique,   # 적을수록 일관됨
        'ft_unique': ft_unique,
    })

    print(f'\n[{cat}] {fname}')
    print(f'  Base 모델 ({base_unique}가지 답변):')
    for j, r in enumerate(base_runs): print(f'    {j+1}. {r}')
    print(f'  Fine-tuned ({ft_unique}가지 답변):')
    for j, r in enumerate(ft_runs):   print(f'    {j+1}. {r}')

avg_base_unique = sum(r['base_unique'] for r in consistency_results) / len(consistency_results)
avg_ft_unique   = sum(r['ft_unique']   for r in consistency_results) / len(consistency_results)
print(f'\n평균 고유 답변 수 (낮을수록 일관됨):')
print(f'  Base:        {avg_base_unique:.1f} / {N_RUNS}')
print(f'  Fine-tuned:  {avg_ft_unique:.1f} / {N_RUNS}')

## 📊 Cell 10: 시각화 — 그래프 생성 (PPT용 PNG)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'NanumGothic'  # 한글 폰트
!apt-get -qq install fonts-nanum 2>/dev/null
!fc-cache -fv 2>/dev/null > /dev/null
import matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Base vs Fine-tuned MobileVLM 성능 비교', fontsize=16, fontweight='bold', y=1.02)

# ── 그래프 1: 포맷 준수율 ──────────────────────────────────────
ax1 = axes[0]
labels = ['Base\n(짧은 프롬프트)', 'Base\n(긴 프롬프트)', 'Fine-tuned\n(짧은 프롬프트)']
values = [base_short_rate, base_long_rate, ft_short_rate]
colors = ['#FF6B6B', '#FFA07A', '#4CAF50']
bars = ax1.bar(labels, values, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.0f}%', ha='center', va='bottom', fontsize=13, fontweight='bold')
ax1.set_ylim(0, 115)
ax1.set_ylabel('포맷 준수율 (%)', fontsize=12)
ax1.set_title('① 포맷 준수율\n(4-class 키워드 포함 1문장)', fontsize=13, fontweight='bold')
ax1.axhline(y=100, color='gray', linestyle='--', alpha=0.5)
ax1.set_facecolor('#FAFAFA')
ax1.spines[['top','right']].set_visible(False)
# 핵심 포인트 주석
ax1.annotate('프롬프트 없이도\n형식 유지!', xy=(2, ft_short_rate),
             xytext=(1.5, ft_short_rate + 8),
             fontsize=10, color='#2E7D32',
             arrowprops=dict(arrowstyle='->', color='#2E7D32'))

# ── 그래프 2: 응답 일관성 ──────────────────────────────────────
ax2 = axes[1]
categories = [r['category'] for r in consistency_results]
base_uniq = [r['base_unique'] for r in consistency_results]
ft_uniq   = [r['ft_unique']   for r in consistency_results]

x = range(len(categories))
w = 0.35
b1 = ax2.bar([i - w/2 for i in x], base_uniq, w, label='Base 모델', color='#FF6B6B', edgecolor='white')
b2 = ax2.bar([i + w/2 for i in x], ft_uniq,   w, label='Fine-tuned', color='#4CAF50', edgecolor='white')
for bar, val in zip(b1, base_uniq): ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05, str(val), ha='center', va='bottom', fontsize=12, fontweight='bold')
for bar, val in zip(b2, ft_uniq):   ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05, str(val), ha='center', va='bottom', fontsize=12, fontweight='bold')
ax2.set_xticks(list(x))
ax2.set_xticklabels(['침대이탈', '이동중', '낙상'], fontsize=12)
ax2.set_ylabel(f'고유 답변 수 (/{N_RUNS}회, 낮을수록 일관됨)', fontsize=11)
ax2.set_title(f'② 응답 일관성\n(동일 이미지 {N_RUNS}회 반복, 고유 답변 수)', fontsize=13, fontweight='bold')
ax2.set_ylim(0, N_RUNS + 1)
ax2.legend(fontsize=11)
ax2.set_facecolor('#FAFAFA')
ax2.spines[['top','right']].set_visible(False)

plt.tight_layout()
chart_path = f'{RESULT_DIR}/metrics_chart.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ 저장: {chart_path}')

## 🖼️ Cell 11: 정성 비교 표 — 이미지 + 두 모델 출력 나란히 (PPT용 PNG)

In [ ]:
from PIL import Image as PILImage
import numpy as np

# 카테고리별 대표 3장 선택
qual_samples = [SAMPLES[0], SAMPLES[3], SAMPLES[6]]  # bed_exit, moving, fall
cat_labels_kr = {'bed_exit': '침대이탈', 'moving': '이동중', 'fall': '낙상'}

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
fig.suptitle('정성적 비교: 동일 이미지, 동일 프롬프트 ("현재 환자의 상태를 보고하세요.")',
             fontsize=14, fontweight='bold', y=1.01)

col_titles = ['실제 이미지', 'Base 모델 출력', 'Fine-tuned 출력']
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)

for row_idx, sample in enumerate(qual_samples):
    cat = sample['category']
    img_path = sample['image']
    base_out = results_base[SAMPLES.index(sample)]['short_output']
    ft_out   = results_ft[SAMPLES.index(sample)]['short_output']
    base_ok  = results_base[SAMPLES.index(sample)]['short_comply']
    ft_ok    = results_ft[SAMPLES.index(sample)]['short_comply']

    # 이미지
    img = PILImage.open(img_path).convert('RGB')
    axes[row_idx][0].imshow(np.array(img))
    axes[row_idx][0].set_ylabel(f'[{cat_labels_kr[cat]}]', fontsize=12, fontweight='bold', rotation=90, labelpad=10)
    axes[row_idx][0].set_xticks([]); axes[row_idx][0].set_yticks([])
    for spine in axes[row_idx][0].spines.values(): spine.set_linewidth(2); spine.set_color('#888')

    # Base 출력 텍스트 박스
    axes[row_idx][1].text(0.5, 0.5, base_out,
        ha='center', va='center', fontsize=12, wrap=True,
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#FFE5E5' if not base_ok else '#E8F5E9', edgecolor='#FF6B6B' if not base_ok else '#4CAF50', linewidth=2))
    axes[row_idx][1].text(0.95, 0.05, '❌ 형식 불일치' if not base_ok else '✅ 형식 준수',
        ha='right', va='bottom', fontsize=10, color='#c62828' if not base_ok else '#2e7d32',
        transform=axes[row_idx][1].transAxes)
    axes[row_idx][1].set_xlim(0,1); axes[row_idx][1].set_ylim(0,1)
    axes[row_idx][1].set_xticks([]); axes[row_idx][1].set_yticks([])
    axes[row_idx][1].set_facecolor('#FFF8F8' if not base_ok else '#F1F8F1')

    # Fine-tuned 출력 텍스트 박스
    axes[row_idx][2].text(0.5, 0.5, ft_out,
        ha='center', va='center', fontsize=12, wrap=True,
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#E8F5E9' if ft_ok else '#FFE5E5', edgecolor='#4CAF50' if ft_ok else '#FF6B6B', linewidth=2))
    axes[row_idx][2].text(0.95, 0.05, '✅ 형식 준수' if ft_ok else '❌ 형식 불일치',
        ha='right', va='bottom', fontsize=10, color='#2e7d32' if ft_ok else '#c62828',
        transform=axes[row_idx][2].transAxes)
    axes[row_idx][2].set_xlim(0,1); axes[row_idx][2].set_ylim(0,1)
    axes[row_idx][2].set_xticks([]); axes[row_idx][2].set_yticks([])
    axes[row_idx][2].set_facecolor('#F1F8F1' if ft_ok else '#FFF8F8')

plt.tight_layout()
table_path = f'{RESULT_DIR}/comparison_table.png'
plt.savefig(table_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ 저장: {table_path}')

## 💾 Cell 12: 결과 리포트 저장 & 다운로드

In [ ]:
import pandas as pd

# ── 수치 요약 리포트 ──────────────────────────────────────────
report_lines = [
    '# Base vs Fine-tuned MobileVLM 평가 리포트',
    '',
    '## 1. 포맷 준수율 (Format Compliance Rate)',
    f'| 모델 & 프롬프트 | 준수율 |',
    f'|---|---|',
    f'| Base (짧은 프롬프트, {len(SHORT_PROMPT)}자) | {base_short_rate:.0f}% |',
    f'| Base (긴 프롬프트, {len(LONG_PROMPT)}자)  | {base_long_rate:.0f}% |',
    f'| Fine-tuned (짧은 프롬프트, {len(SHORT_PROMPT)}자) | **{ft_short_rate:.0f}%** |',
    '',
    '## 2. 응답 일관성 (평균 고유 답변 수 / 5회)',
    f'| 모델 | 평균 고유 답변 수 |',
    f'|---|---|',
    f'| Base 모델 | {avg_base_unique:.1f} / {N_RUNS} |',
    f'| Fine-tuned | **{avg_ft_unique:.1f} / {N_RUNS}** |',
    '',
    '## 3. 결론',
    '- Fine-tuned 모델은 **짧은 프롬프트만으로도** 도메인 형식을 일관되게 준수',
    '- Base 모델은 긴 프롬프트 없이 형식 붕괴 발생 → 엣지 배포 부적합',
    '- Fine-tuned 모델: 빠른 추론 (짧은 프롬프트) + 안정적 출력 = NPU 엣지 배포 최적',
]

report_path = f'{RESULT_DIR}/evaluation_report.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(report_lines))

# ── 개별 결과 CSV ──────────────────────────────────────────────
df = pd.DataFrame([
    {'image': r['image'], 'category': r['category'],
     'base_short': r['short_output'], 'base_short_comply': r['short_comply'],
     'base_long': r['long_output'],   'base_long_comply':  r['long_comply'],
     'ft_short': results_ft[i]['short_output'], 'ft_short_comply': results_ft[i]['short_comply']}
    for i, r in enumerate(results_base)
])
csv_path = f'{RESULT_DIR}/detailed_results.csv'
df.to_csv(csv_path, index=False, encoding='utf-8-sig')

print('✅ 모든 결과 저장 완료:')
print(f'  {chart_path}')
print(f'  {table_path}')
print(f'  {report_path}')
print(f'  {csv_path}')
print()
print('\n'.join(report_lines))

# ── 로컬 다운로드 ──────────────────────────────────────────────
from google.colab import files
print('\n📥 로컬 다운로드 시작...')
files.download(chart_path)
files.download(table_path)
files.download(report_path)